In [1]:
from fastai.collab import *
from fastai.tabular.all import *

In [2]:
movies_path = "./data/movies.csv"
ratings_path = "./data/ratings.csv"

In [3]:
movies_df = pd.read_csv(movies_path)
ratings_df = pd.read_csv(ratings_path)

In [4]:
ratings_df = ratings_df.merge(movies_df)

In [ ]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


: 

In [6]:
movie_value_counts = ratings_df.movieId.value_counts()
movie_value_counts

movieId
356       81491
318       81482
296       79672
593       74127
2571      72674
          ...  
150198        1
150206        1
150212        1
150304        1
150088        1
Name: count, Length: 59047, dtype: int64

In [7]:
ratings_df = ratings_df[
    ratings_df['movieId'].isin(movie_value_counts[movie_value_counts >= 5].index)
]

In [8]:
len(ratings_df)

24945870

In [9]:
len(ratings_df["userId"].unique()), len(ratings_df["movieId"].unique())

(162541, 32720)

In [10]:
dls = CollabDataLoaders.from_df(ratings_df, item_name="title", bs=50000)

In [11]:
dls.show_batch()

,userId,title,rating
0,52801,Home Alone (1990),3.0
1,33379,"Nightmare Before Christmas, The (1993)",3.0
2,115645,"Grand Budapest Hotel, The (2014)",4.0
3,112846,Schindler's List (1993),3.0
4,94869,S.F.W. (1994),3.0
5,146652,Beverly Hills Cop (1984),3.0
6,68660,SLC Punk! (1998),3.0
7,119876,Apollo 13 (1995),5.0
8,56661,"Clockwork Orange, A (1971)",1.5
9,91184,City of God (Cidade de Deus) (2002),5.0


In [12]:
learn = collab_learner(dls, y_range=(0, 5.5), n_factors=300)

In [27]:
learn.fit_one_cycle(30, 1e-3, wd=1e-1)

epoch,train_loss,valid_loss,time
0,0.328162,0.587957,00:47
1,0.325708,0.587928,00:43
2,0.321506,0.588217,00:44
3,0.314447,0.589068,00:44
4,0.304868,0.590531,00:44
5,0.293586,0.592597,00:44
6,0.281609,0.594896,00:44
7,0.269828,0.597318,00:44
8,0.259192,0.599923,00:45
9,0.249322,0.602415,00:45


KeyboardInterrupt: 

In [24]:
learn.export("./movie_recomendation_system.pkl")

In [28]:
learn = load_learner("./movie_recomendation_system.pkl", cpu=True)

/home/aidev4/PycharmProjects/Datascience/venv/lib/python3.12/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


In [29]:
preds, targ = learn.get_preds(dl=dls.valid)

In [30]:
from fastai.metrics import rmse
rmse(preds, targ)

TensorBase(0.7668)

In [34]:
movie_bias = learn.model.i_bias.weight.squeeze()
idxs = movie_bias.argsort(descending=False)[:20]
[dls.classes['title'][i] for i in idxs]

['From Justin to Kelly (2003)',
 'SuperBabies: Baby Geniuses 2 (2004)',
 'Glitter (2001)',
 'Disaster Movie (2008)',
 'Son of the Mask (2005)',
 "Barney's Great Adventure (1998)",
 'Gigli (2003)',
 'Pokémon Heroes (2003)',
 'Epic Movie (2007)',
 'House of the Dead, The (2003)',
 'Home Alone 4 (2002)',
 'Bratz: The Movie (2007)',
 'Turbo: A Power Rangers Movie (1997)',
 'Pokemon 4 Ever (a.k.a. Pokémon 4: The Movie) (2002)',
 'Dragonball Evolution (2009)',
 'Battlefield Earth (2000)',
 'Faces of Death 4 (1990)',
 'Alone in the Dark (2005)',
 'Faces of Death 6 (1996)',
 'Jack and Jill (2011)']

In [35]:
user_factors = learn.model.u_weight.weight.detach()
item_factors = learn.model.i_weight.weight.detach()

user_bias = learn.model.u_bias.weight.detach().squeeze()
item_bias = learn.model.i_bias.weight.detach().squeeze()

In [36]:
user_map = dict(enumerate(dls.classes['userId']))
user_to_idx = {v: k for k, v in user_map.items()}

movie_map = dict(enumerate(dls.classes['title']))
idx_to_movie = movie_map

In [37]:
def fast_recommend(user_id, n=10):
    user_idx = user_to_idx[user_id]
    u_vec = user_factors[user_idx]

    scores = (item_factors @ u_vec) + item_bias + user_bias[user_idx]

    rated_books = ratings_df[ratings_df['userId'] == user_id]['title']
    rated_set = set(rated_books)

    for idx, book in idx_to_movie.items():
        if book in rated_set:
            scores[idx] = -1e9

    top_idx = torch.topk(scores, n).indices
    return [idx_to_movie[i.item()] for i in top_idx]

In [38]:
fast_recommend(17166)

['Casablanca (1942)',
 'Citizen Kane (1941)',
 'Seven Samurai (Shichinin no samurai) (1954)',
 'Lawrence of Arabia (1962)',
 'Rear Window (1954)',
 'To Kill a Mockingbird (1962)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Vertigo (1958)',
 "Singin' in the Rain (1952)",
 'Bridge on the River Kwai, The (1957)']

In [39]:
import torch
import torch.nn.functional as F

# Normalize item embeddings
item_factors_norm = F.normalize(item_factors, p=2, dim=1)

In [40]:
len(item_factors)

32685

In [41]:
idx_to_movie2 = {v: k for k, v in movie_map.items()}


In [42]:
item_factors_norm

tensor([[-0.0641, -0.0147,  0.0894,  ...,  0.0161,  0.0224, -0.0106],
        [ 0.0550, -0.0129, -0.0001,  ..., -0.0121, -0.0337,  0.0999],
        [ 0.0179, -0.0049,  0.0353,  ...,  0.0287,  0.0097, -0.0312],
        ...,
        [-0.0434, -0.0357, -0.0372,  ...,  0.0143, -0.0531,  0.0396],
        [-0.0897, -0.0865, -0.0215,  ...,  0.0490,  0.0868,  0.0497],
        [ 0.0600,  0.0409,  0.0743,  ...,  0.0123, -0.0063, -0.0105]])

In [61]:
movie_name = "Vikramarkudu (2006)"
movie_idx = idx_to_movie2[movie_name]
query_vector = item_factors_norm[movie_idx]


similarities = torch.matmul(item_factors_norm, query_vector)


top_k = 10

# Get top 11 because the movie itself will be most similar (similarity = 1)
values, indices = torch.topk(similarities, top_k + 1)

# Remove the movie itself
similar_movies = indices[indices != movie_idx][:top_k]

In [62]:
for idx in similar_movies:
    print(movie_map[int(idx)])

Immigrant, The (1917)
Destiny (a.k.a. Between Two Worlds) (Der müde Tod) (1921)
Night and Fog (Nuit et brouillard) (1955)
Rififi (Du rififi chez les hommes) (1955)
Hearts and Minds (1974)
Trou, Le (Hole, The) (Night Watch, The) (1960)
Rack, The (1956)
Hands Off the Loot (Touchez pas au grisbi) (1954)
Red Beard (Akahige) (1965)
Vitelloni, I (a.k.a. The Young and the Passionate) (1953)


In [60]:
for k, v in movie_map.items():
    if "vikram" in v.lower():
        print(v, k)

Vikram Vedha (2017) 30888
Vikramarkudu (2006) 30889
